In [1]:
import pandas as pd
import geopandas as gpd
import libpysal
from esda.moran import Moran
from esda.moran import Moran_Local
import numpy as np
from pyprojroot import here

In [2]:
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results" / "esda"

In [3]:
df = pd.read_csv(PROCESSED_DATA_DIR/'RELMS_index.csv', dtype={'CVEGEO': str})
adj = pd.read_csv(INTERIM_DATA_DIR/'adj_combined.csv', index_col=0, dtype=str)
adj.index = adj.index.astype(str).str.zfill(5)
adj.columns = adj.columns.astype(str).str.zfill(5)

In [4]:
df['CVEGEO'] = df['CVEGEO'].astype(str).str.zfill(5)
df_aligned = df.set_index('CVEGEO').loc[adj.index].reset_index()

### Loading contiguity matrix

In [5]:
adj_num = adj.astype(int)
edges = pd.DataFrame(
    [(i, j) for i in adj_num.index for j in adj_num.columns if adj_num.loc[i, j] > 0],
    columns=['focal', 'neighbor']
)
edges['weight'] = 1

In [6]:
w = libpysal.weights.W.from_adjlist(edges, focal_col='focal', neighbor_col='neighbor', weight_col='weight')
w.transform = 'r'

### Moran's I

In [7]:
y = df_aligned.rename(columns={'index': 'CVEGEO'}).set_index('CVEGEO').loc[w.id_order, 'RELM_index'].values

mi = Moran(y, w)

In [8]:
results = pd.DataFrame([{
    'I_statistic': mi.I,
    'Expected_I': mi.EI,
    'z_score': mi.z_norm,
    'p_value_normal': mi.p_norm,
    'p_value_simulated': mi.p_sim,
    'permutations': mi.permutations,
    'interpretation': f"{'Positive' if mi.I > 0 else 'Negative'} spatial autocorrelation ({'significant' if mi.p_sim < 0.05 else 'not significant'} at 5%)"
}])

results.to_csv(RESULTS_DATA_DIR/'morans_i_RELM.csv', index=False)
print(results.T)

                                                                   0
I_statistic                                                 0.572968
Expected_I                                                 -0.000404
z_score                                                    47.194727
p_value_normal                                                   0.0
p_value_simulated                                              0.001
permutations                                                     999
interpretation     Positive spatial autocorrelation (significant ...


### LISA

In [9]:
lisa = Moran_Local(y, w, permutations=999)

In [10]:
quad_labels = {1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}

In [11]:
sig = lisa.p_sim < 0.05

df_lisa = pd.DataFrame({
    'CVEGEO': w.id_order,
    'local_I': lisa.Is,
    'p_sim': lisa.p_sim,
    'quadrant': [quad_labels[q] for q in lisa.q],
    'cluster': [quad_labels[q] if sig[i] else 'ns' for i, q in enumerate(lisa.q)]
})

In [12]:
print(df_lisa['cluster'].value_counts())
df_lisa.to_csv(RESULTS_DATA_DIR/'LISA_RELM.csv', index=False)

cluster
ns    1543
HH     530
LL     354
HL      31
LH      20
Name: count, dtype: int64
